# Pork rasher packaging inspection: full pipeline
Detect packaging defects with YOLO, then decide tray PASS/FAIL. Run with a **GPU runtime** and a Colab secret
named `ROBOFLOW_API_KEY`.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/ElijahKhoury1/pork-rasher-qc.git"   
import os
if not os.path.isdir("pork-rasher-qc"):
    !git clone {REPO_URL} pork-rasher-qc
%cd pork-rasher-qc
!pip install -q ultralytics roboflow pandas matplotlib
!pip install -q -e .

In [ ]:
import torch, sys, platform
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())
SEED = 0

## 2. Download the dataset (Version 4, YOLO format)

In [ ]:
from google.colab import userdata
from rasher_qc import data

DATASET = data.download_roboflow(userdata.get("ROBOFLOW_API_KEY"), version=4, location="datasets/pork-rasher")
DATA_YAML = str(DATASET / "data.yaml")
print(DATASET)

## 3. Check the data
Image counts, missing labels, class balance, and **train/test leakage** (augmented copies of one frame in
several splits would inflate test scores).

In [ ]:
import pandas as pd
print(data.validate_dataset(DATASET))
counts = pd.DataFrame(data.count_classes(DATASET)).fillna(0).astype(int)
display(counts)
leaks = data.find_cross_split_leakage(DATASET)
print(f"Frames appearing in more than one split: {len(leaks)}")

In [ ]:
# Look at a few annotated images
import random, glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
random.seed(SEED)
files = random.sample(data.list_images(DATASET, "train"), 6)
fig, axs = plt.subplots(2, 3, figsize=(12, 6))
for ax, f in zip(axs.ravel(), files):
    ax.imshow(mpimg.imread(f)); ax.set_title(f.name[:28], fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Train: baseline and alternatives
Baseline = `yolov8n` at 640 px. Alternatives vary model size and input resolution. Selection uses the
**validation** split only.

In [ ]:
from rasher_qc import train, evaluate

EXPERIMENTS = {
    "baseline_n_640": dict(weights="yolov8n.pt", imgsz=640),
    "s_640":          dict(weights="yolov8s.pt", imgsz=640),
    "n_416":          dict(weights="yolov8n.pt", imgsz=416),
}
EPOCHS = 50          # lower (e.g. 5) for a quick smoke test

best = {}
for name, cfg in EXPERIMENTS.items():
    best[name] = train.train(DATA_YAML, epochs=EPOCHS, seed=SEED, name=name, **cfg)
    print(name, "->", best[name])

## 5. Compare on the validation split (model selection)

In [ ]:
rows = []
for name, cfg in EXPERIMENTS.items():
    m = evaluate.detection_metrics(best[name], DATA_YAML, split="val", imgsz=cfg["imgsz"])
    imgs = data.list_images(DATASET, "valid")[:20]
    b = evaluate.benchmark_latency(best[name], imgs, imgsz=cfg["imgsz"])
    rows.append({"model": name, "mAP50": m["map50"], "mAP50-95": m["map50_95"],
                 "precision": m["precision"], "recall": m["recall"],
                 "latency_ms": b["mean_ms"], "fps": b["fps"], "params_M": b["params_m"], "GFLOPs": b["gflops"]})
results = pd.DataFrame(rows).set_index("model").round(3)
display(results)
results.to_csv("experiment_results.csv")
SELECTED = results["mAP50-95"].idxmax()
print("Selected on validation mAP50-95:", SELECTED)

## 6. Final test-set evaluation (selected model only, run once)
Detection metrics, per-class AP, and the tray-level PASS/FAIL result.

In [ ]:
cfg = EXPERIMENTS[SELECTED]
final = evaluate.detection_metrics(best[SELECTED], DATA_YAML, split="test", imgsz=cfg["imgsz"])
print({k: round(v, 3) for k, v in final.items() if k != "per_class_map50_95"})
display(pd.Series(final["per_class_map50_95"], name="AP50-95").round(3).to_frame())

# Tray-level decision. Change FAIL_CLASSES if some classes should not cause a reject (None = all).
FAIL_CLASSES = None
tray = evaluate.tray_level_metrics(best[SELECTED], DATASET, split="test", conf=0.25,
                                   imgsz=cfg["imgsz"], fail_classes=FAIL_CLASSES)
print(tray)

## 7. Sensitivity: confidence threshold
The threshold trades missed defects (recall) against false rejects (precision).

In [ ]:
sens = []
for c in [0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6]:
    t = evaluate.tray_level_metrics(best[SELECTED], DATASET, split="test", conf=c,
                                    imgsz=cfg["imgsz"], fail_classes=FAIL_CLASSES)
    sens.append({"conf": c, "precision": t["precision"], "recall": t["recall"], "f1": t["f1"]})
sens = pd.DataFrame(sens).set_index("conf")
display(sens.round(3))
sens.plot(marker="o", title="Tray-level metrics vs confidence threshold"); plt.grid(alpha=.3); plt.show()

## 8. Inspect individual trays and look at failures

In [ ]:
from ultralytics import YOLO
from rasher_qc.infer import inspect_tray
model = YOLO(str(best[SELECTED]))
for img in data.list_images(DATASET, "test")[:5]:
    r = inspect_tray(model, img, conf=0.25, imgsz=cfg["imgsz"], fail_classes=FAIL_CLASSES)
    print(img.name[:30], r["status"], r["counts"])

In [ ]:
# Ultralytics saves a confusion matrix and PR curves on validation runs
from IPython.display import Image, display
val_dir = sorted(glob.glob("runs/detect/val*"), key=os.path.getmtime)
print(val_dir[-1] if val_dir else "no val folder found")
for f in ["confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    p = os.path.join(val_dir[-1], f) if val_dir else ""
    if os.path.exists(p): display(Image(p, width=600))

## 9. Next steps for the report
- Look at false positives and misses on the test split: reflections, occlusion, overlapping meat.
- Check whether results change when augmentation (saturation/exposure) is removed.
- Save `experiment_results.csv`, the threshold table and key figures into the repo for the write-up.
- Commit with `git add`, `git commit`, `git push` from the Colab terminal or download the notebook.